In [1]:
import glob
import matplotlib.pyplot as plt
import numpy as np
import cv2
import re
import pandas as pd

from functions import *

In [2]:
# Input
folder_in_main = '/media/joris/rootfs/home/joris/data/kiwibes2026/20260706kiwibes/bes21-30-joris'

In [3]:
# Prepare machine and segmentation model
predictor = prepare_model(verbose = False)

In [4]:
! rm -r bes*

/bin/bash: /home/joris/miniconda3/envs/pytorch/lib/libtinfo.so.6: no version information available (required by /bin/bash)


In [5]:
! ls

/bin/bash: /home/joris/miniconda3/envs/pytorch/lib/libtinfo.so.6: no version information available (required by /bin/bash)
dev  flow.ipynb  functions.py  __pycache__  README.md  results.csv


In [ ]:
results = []

fig, axes = plt.subplots(1, 3, figsize=(9, 4))

for folder_in_berry in glob.glob(f'{folder_in_main}/bes3*'):

    berry_name = folder_in_berry.split('/')[-1]
    print()
    print(berry_name)

    if not os.path.exists(berry_name):
        os.mkdir(berry_name)

    ellipses_berry = []

    for file_in in glob.glob(f'{folder_in_berry}/*'):

        print('  ', re.sub(folder_in_main, '', file_in))

        img = cv2.imread(file_in)
        grid_spacing = find_grid_spacing(img, visualize=False)
        mask = make_mask(img, predictor, visualize=False)

        if mask is not None:
            ellipse = fit_ellipse(mask)
            ellipses_berry.append(ellipse)

        # Clear each subplot explicitly
        for ax in axes:
            ax.clear()
            ax.axis('off')

        # Subplot 1
        axes[0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        axes[0].axis('off')

        # Subplot 2
        if mask is not None:
            axes[1].imshow(mask)
        else:
            axes[1].imshow(np.zeros(img.shape))
        axes[1].axis('off')

        # Subplot 3
        if mask is not None:
            plot_mask_ellipse(mask, ellipse, ax=axes[2])
        else:
            axes[2].imshow(np.zeros(img.shape))
        axes[2].axis('off')

        plt.tight_layout()

        file_plot = f'{berry_name}/{file_in.split("/")[-1]}'
        fig.savefig(file_plot)


    # Select three main radii (mm)
    r1 = np.max([ellipse['semi_major'] for ellipse in ellipses_berry])*50/grid_spacing
    r2 = np.max([ellipse['semi_minor'] for ellipse in ellipses_berry])*50/grid_spacing
    r3 = np.min([ellipse['semi_minor'] for ellipse in ellipses_berry])*50/grid_spacing
    
    # Calculate volume (mm**3)
    vol = 4*np.pi/3*r1*r2*r3
    
    # Append result
    result = {
        'berry_name': berry_name,
        'grid_spacing': grid_spacing,
        'r1 (mm)': r1,
        'r2 (mm)': r2,
        'r3 (mm)': r3,
        'volume (mm**3)': vol
    }
    results.append(result)

plt.close(fig)

pd.DataFrame(results).to_csv('results.csv')


bes30
   /bes30/20260709_143736.jpg
   /bes30/20260709_143747.jpg


In [ ]:
pd.DataFrame(results)

# EXP

In [ ]:
STOP

In [ ]:
file_in = f'{folder_in_main}/bes30/20260709_142545.jpg'
img = cv2.imread(file_in)
# make_mask(img, predictor, step = 500, visualize = True)


step = 500
visualize = False

# Generate query points for SAM
queries = np.array(list(itertools.product(
    np.arange(step//2, img.shape[1], step),
    np.arange(step//2, img.shape[0], step), 
)))

# Apply SAM2
predictor.set_image(img)
candidates = []

for point in queries:
    input_point = np.array([point])
    input_label = np.array([1])
    masks, scores, logits = predictor.predict(
        point_coords=input_point,
        point_labels=input_label,
        multimask_output=True,
    )

    # Some selection
    for mask in masks:

        if visualize:
            plt.figure()
            plt.subplot(121)
            plt.imshow(img)
            plt.scatter([input_point[0,0]], [input_point[0,1]])
            plt.subplot(122)
            plt.imshow(mask)
            plt.show()

        # Size
        mask_size_min = 100000
        mask_size_max = 1000000
        mask_size = np.sum(mask)
        if mask_size < mask_size_min or mask_size > mask_size_max:
            if visualize:
                print('Rejected because of size.')
            break
    
        # Centeredness
        mask_cx_min = mask.shape[0]*0.25
        mask_cx_max = mask.shape[0]*0.75
        mask_cx = np.mean(np.where(mask)[0])
        if mask_cx < mask_cx_min or mask_cx > mask_cx_max:
            if visualize:
                print('Rejected because of centeredness.')
            break
        
        # mask_cy_min = mask.shape[1]*0.25
        # mask_cy_max = mask.shape[1]*0.75
        # mask_cy = np.mean(np.where(mask)[1])
        # if mask_cy < mask_cy_min or mask_cy > mask_cy_max:
        #     if visualize:
        #         print('Rejected because of centeredness.')
        #     break

        # Outer dimensions
        mask_xrange_min = 500
        mask_xrange_max = 2000
        mask_xrange = max(np.where(mask)[0])-min(np.where(mask)[0])
        if mask_xrange < mask_xrange_min or mask_xrange > mask_xrange_max:
            if visualize:
                print('Rejected because of outer dimensions.')
            break
        
        mask_yrange_min = 500
        mask_yrange_max = 1500
        mask_yrange = max(np.where(mask)[1])-min(np.where(mask)[1])
        if mask_yrange < mask_yrange_min and mask_yrange > mask_yrange_max:
            if visualize:
                print('Rejected because of outer dimensions.')
            break

        # Greenness
        # mask_green_min = 0.8
        mask_coords = np.array(np.where(mask)).transpose()
        mask_rgb = np.array([img[x, y] for x, y in  mask_coords])
        # mask_green = (mask_rgb[:,0] < mask_rgb[:,1]) & (mask_rgb[:,2] < mask_rgb[:,1]) 
        # mask_green = mask_rgb[:,0] < mask_rgb[:,1]
        mask_green = 2*mask_rgb[:,1] - mask_rgb[:,0] - mask_rgb[:,2]
        mask_greenness = np.mean(mask_green)
        # if mask_greenness < mask_green_min:
        #     if visualize:
        #         print(f'Rejected because of greenness: {mask_greenness}')
        #     break

        # Filledness
        fill_rate_min = 0.8
        hull = convex_hull_image(mask)
        fill_rate = np.sum(hull & mask.astype(bool))/np.sum(hull)
        if fill_rate < fill_rate_min:
            if visualize:
                print(f'Rejected because of filledness: {fill_rate}')
            break

        # All test passed        
        candidate = {
            'mask': mask,
            'greenness': mask_greenness,
            'fill_rate': fill_rate
        }
        candidates.append(candidate)


for candidate in candidates:

    print(candidate['greenness'])
    print(candidate['fill_rate'])
    plt.figure()
    plt.imshow(candidate['mask'])
    plt.show()

In [ ]:
predictor.set_image(img)
input_point = np.array([point])
input_label = np.array([1])
masks, scores, logits = predictor.predict(
    point_coords=input_point,
    point_labels=input_label,
    multimask_output=True,
)

In [ ]:
for mask in masks:
    plt.figure()
    plt.subplot(121)
    plt.imshow(img)
    plt.scatter([input_point[0,0]], [input_point[0,1]])
    plt.subplot(122)
    plt.imshow(mask)
    plt.show()

In [ ]:
mask_coords = np.array(np.where(mask)).transpose()
mask_rgb = np.array([img[x, y] for x, y in  mask_coords])
mask_green = (mask_rgb[:,0] < mask_rgb[:,1]) & (mask_rgb[:,2] < mask_rgb[:,1])
mask_greenness = np.mean(mask_green)
mask_greenness

In [ ]:
plt.hist(mask_rgb[:,0], color='red')
plt.hist(mask_rgb[:,1], color='green')
plt.hist(mask_rgb[:,2], color='blue')

In [ ]:
mask

In [ ]:
# if mask is not None:
#     ellipse = fit_ellipse(mask)
#     ellipses_berry.append(ellipse)

# # Clear each subplot explicitly
# for ax in axes:
#     ax.clear()
#     ax.axis('off')

# # Subplot 1
# axes[0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
# axes[0].axis('off')

# # Subplot 2
# if mask is not None:
#     axes[1].imshow(mask)
# else:
#     axes[1].imshow(np.zeros(img.shape))
# axes[1].axis('off')

# # Subplot 3
# if mask is not None:
#     plot_mask_ellipse(mask, ellipse, ax=axes[2])
# else:
#     axes[2].imshow(np.zeros(img.shape))
# axes[2].axis('off')

# plt.tight_layout()

# file_plot = f'{berry_name}/{file_in.split("/")[-1]}'
# # fig.savefig(file_plot)


plt.close(fig)